In [1]:
import numpy as np
from numpy import unique
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix
from collections import defaultdict

class KMeans:
    def __init__(self, n_clusters=3, max_iter=300, random_state=93, n_init=10):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.random_state = random_state
        self.n_init = n_init
        self.cluster_centers_ = None
        self.labels_ = None
        self.inertia_ = None

    def _euclidean_distance(self, x1, x2):
        return np.sqrt(np.sum((x1 - x2) ** 2, axis=1))

    def _init_centroids(self, X):
        np.random.seed(self.random_state)
        n_samples, n_features = X.shape
        indices = np.random.choice(n_samples, self.n_clusters, replace=False)
        centroids = X[indices]
        return centroids

    def _assign_clusters(self, X, centroids):
        n_samples = X.shape[0]
        labels = np.zeros(n_samples, dtype=int)
        for i in range(n_samples):
            distances = self._euclidean_distance(X[i], centroids)
            labels[i] = np.argmin(distances)
        return labels

    def _update_centroids(self, X, labels):
        n_features = X.shape[1]
        centroids = np.zeros((self.n_clusters, n_features))
        for k in range(self.n_clusters):
            cluster_samples = X[labels == k]
            if len(cluster_samples) > 0:
                centroids[k] = np.mean(cluster_samples, axis=0)
        return centroids

    def _calculate_inertia(self, X, labels, centroids):
        inertia = 0.0
        for k in range(self.n_clusters):
            cluster_samples = X[labels == k]
            if len(cluster_samples) > 0:
                inertia += np.sum((cluster_samples - centroids[k]) ** 2)
        return inertia

    def fit(self, X):
        X = np.array(X)
        n_samples, n_features = X.shape
        best_inertia = np.inf
        best_centroids = None
        best_labels = None

        for _ in range(self.n_init):
            centroids = self._init_centroids(X)
            prev_centroids = centroids

            for _ in range(self.max_iter):
                labels = self._assign_clusters(X, centroids)
                centroids = self._update_centroids(X, labels)
                if np.allclose(centroids, prev_centroids):
                    break
                prev_centroids = centroids

            inertia = self._calculate_inertia(X, labels, centroids)
            if inertia < best_inertia:
                best_inertia = inertia
                best_centroids = centroids
                best_labels = labels

        self.cluster_centers_ = best_centroids
        self.labels_ = best_labels
        self.inertia_ = best_inertia

    def predict(self, X):
        X = np.array(X)
        return self._assign_clusters(X, self.cluster_centers_)



iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

df = pd.DataFrame(X, columns=feature_names)
df['target'] = y

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

k = 3
kmeans = KMeans(n_clusters=k, random_state=43, n_init=10)
kmeans.fit(X_scaled)

cluster_labels = kmeans.labels_
centroids = kmeans.cluster_centers_
inertia = kmeans.inertia_

print(f"\n===== K-Means聚类结果（K={k}） =====")
print(f"最终质心（标准化后）：\n{centroids}")
print(f"惯性值：{inertia:.4f}")
print(f"各簇样本数量：\n{pd.Series(cluster_labels).value_counts()}")

# ===================== 【关键修复】标签匹配函数 =====================
def map_clusters(true_labels, pred_labels):
    cm = confusion_matrix(true_labels, pred_labels)
    mapping = np.zeros_like(unique(true_labels))
    for i in range(len(cm)):
        new_label = np.argmax(cm[i])
        mapping[i] = new_label
    map_pred=defaultdict(int)
    for idx,i in enumerate(mapping):
        map_pred[i]=idx
    return map_pred

pred_matched = map_clusters(y, cluster_labels)
# ===================== 修复后的正确率计算 =====================
# 1. 先用映射字典，把原始的聚类标签转换成和真实标签对齐的新标签
y_pred_mapped = np.array([pred_matched[label] for label in cluster_labels])
# 2. 再用转换后的预测标签和真实标签比较，计算准确率
accuracy = np.sum(y_pred_mapped == y) / len(y)
# ===================== 计算正确正确率 =====================
print("\n===== 聚类正确率（匹配标签后）=====")
print(f"准确率：{accuracy:.4f}")






===== K-Means聚类结果（K=3） =====
最终质心（标准化后）：
[[-1.32765367 -0.373138   -1.13723572 -1.11486192]
 [-0.81623084  1.31895771 -1.28683379 -1.2197118 ]
 [ 0.57100359 -0.37176778  0.69111943  0.66315198]]
惯性值：191.0247
各簇样本数量：
2    96
1    33
0    21
Name: count, dtype: int64

===== 聚类正确率（匹配标签后）=====
准确率：0.6667
